In [1]:
import sys
print(sys.executable)
import gpaw, ase
print("gpaw", gpaw.__version__, "| ase", ase.__version__)

/home/buyer/miniconda3/envs/dft/bin/python
gpaw 25.7.0 | ase 3.29.0


In [2]:
from ase.build import bulk
from gpaw import GPAW, PW

# build the silicon crystal: diamond structure, experimental lattice constant
si = bulk('Si', 'diamond', a=5.431)
print(si)

# attach the DFT engine: PBE functional, 300 eV cutoff, 4x4x4 k-points
si.calc = GPAW(xc='PBE', mode=PW(300), kpts=(4, 4, 4), txt='si_first.txt')

# compute the total energy — the first real quantum mechanics you've run
energy = si.get_potential_energy()
print("total energy:", round(energy, 4), "eV")

Atoms(symbols='Si2', pbc=True, cell=[[0.0, 2.7155, 2.7155], [2.7155, 0.0, 2.7155], [2.7155, 2.7155, 0.0]])
total energy: -10.7877 eV


In [3]:
# Step 3a: energy vs. plane-wave cutoff — does the energy plateau?
cutoffs = [200, 250, 300, 350, 400, 450, 500]
energies = []
for ec in cutoffs:
    si.calc = GPAW(xc='PBE', mode=PW(ec), kpts=(4, 4, 4), txt=None)
    e = si.get_potential_energy()
    energies.append(e)
    print(f"cutoff {ec} eV -> energy {e:.4f} eV")

cutoff 200 eV -> energy -10.7793 eV
cutoff 250 eV -> energy -10.7854 eV
cutoff 300 eV -> energy -10.7877 eV
cutoff 350 eV -> energy -10.7885 eV
cutoff 400 eV -> energy -10.7890 eV
cutoff 450 eV -> energy -10.7894 eV
cutoff 500 eV -> energy -10.7897 eV


In [4]:
# Step 3b: energy vs. k-point grid — same plateau test, cutoff frozen at 400
kgrids = [2, 3, 4, 5, 6, 8]
energies_k = []
for k in kgrids:
    si.calc = GPAW(xc='PBE', mode=PW(400), kpts=(k, k, k), txt=None)
    e = si.get_potential_energy()
    energies_k.append(e)
    print(f"k-grid {k}x{k}x{k} -> energy {e:.4f} eV")

k-grid 2x2x2 -> energy -10.6288 eV
k-grid 3x3x3 -> energy -10.1704 eV
k-grid 4x4x4 -> energy -10.7890 eV
k-grid 5x5x5 -> energy -10.7266 eV
k-grid 6x6x6 -> energy -10.7916 eV
k-grid 8x8x8 -> energy -10.7917 eV


In [5]:
import numpy as np

# Step 4: energy vs. lattice constant — the minimum is DFT's predicted lattice constant
a_values = np.linspace(5.30, 5.55, 8)
E_a = []
for a in a_values:
    si_a = bulk('Si', 'diamond', a=float(a))
    si_a.calc = GPAW(xc='PBE', mode=PW(400), kpts=(6, 6, 6), txt=None)
    e = si_a.get_potential_energy()
    E_a.append(e)
    print(f"a = {a:.3f} A -> E = {e:.4f} eV")

a = 5.300 A -> E = -10.6816 eV
a = 5.336 A -> E = -10.7259 eV
a = 5.371 A -> E = -10.7591 eV
a = 5.407 A -> E = -10.7819 eV
a = 5.443 A -> E = -10.7948 eV
a = 5.479 A -> E = -10.7985 eV
a = 5.514 A -> E = -10.7936 eV
a = 5.550 A -> E = -10.7805 eV


In [6]:
# fit a parabola near the minimum to locate it precisely
coeffs = np.polyfit(a_values[3:7], E_a[3:7], 2)   # quadratic through the 4 points around the bottom
a_min = -coeffs[1] / (2 * coeffs[0])
print(f"DFT-PBE lattice constant: {a_min:.4f} A")
print(f"experiment:               5.4310 A")
print(f"error: {100*(a_min - 5.431)/5.431:+.2f}%")

DFT-PBE lattice constant: 5.4762 A
experiment:               5.4310 A
error: +0.83%


In [7]:
# Step 5: silicon's bandgap with PBE
from ase.dft.bandgap import bandgap

si_gap = bulk('Si', 'diamond', a=5.431)   # experimental lattice constant
si_gap.calc = GPAW(xc='PBE', mode=PW(400), kpts=(8, 8, 8), txt=None)
si_gap.get_potential_energy()             # run the ground state

gap, p1, p2 = bandgap(si_gap.calc)
print(f"PBE bandgap: {gap:.3f} eV")
print(f"experimental gap: 1.17 eV")
print(f"underestimation: {100*(1.17 - gap)/1.17:.0f}%")

PBE bandgap: 0.814 eV
experimental gap: 1.17 eV
underestimation: 30%


## Phase 0 findings — DFT validated on silicon
- **Stack:** ASE + GPAW 25.7 in WSL/Ubuntu (no native Windows GPAW).
- **Converged settings:** PW cutoff 400 eV, k-grid 6×6×6 (energy plateaus; sub-meV steps).
- **Silent-lie demo:** a 3×3×3 k-grid gives an energy off by ~0.6 eV with **no warning** — unconverged DFT lies silently. Also: odd/even k-grids converge on different tracks; compare like with like.
- **Lattice constant:** DFT-PBE 5.476 Å vs. experiment 5.431 Å → **+0.83%**, textbook PBE overestimation. Setup validated.
- **Bandgap:** PBE 0.814 eV vs. experiment 1.17 eV → **−30%** (and truly ~−50%: our discrete k-grid misses silicon's indirect CBM, so 0.814 overstates the PBE gap). The famous underestimation, reproduced firsthand — same error class as the ~36% in the perovskite screening literature. Phase 1 quantifies it properly.